# Version 6.2 : XGBoost Quick Wins

**3 Optimisations Rapides** :
1. 🔢 **Polynomial Features** (top 5 features) → +0.01-0.02
2. 🔗 **Feature Interactions** (top 10 × top 10) → +0.01
3. 🎲 **Bagging Ensemble** (5 seeds) → +0.005

**Temps estimé** : 2-3h  
**Gain attendu** : +0.025-0.035  
**Target** : C-index **0.76-0.78**

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from itertools import combinations
from sklearn.preprocessing import PolynomialFeatures

from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported")

✓ Libraries imported


## 2. Configuration

In [2]:
# Best parameters (from V6.1 or manual tuning)
BEST_XGB_PARAMS = {
    'objective': 'survival:cox',
    'eval_metric': 'cox-nloglik',
    'tree_method': 'hist',
    'learning_rate': 0.05,
    'max_depth': 6,
    'min_child_weight': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
}

# Number of models for bagging
N_BAGGING_MODELS = 5

print("✓ Configuration loaded")

✓ Configuration loaded


## 3. Data Loading & Base Features

In [3]:
# Same feature engineering as V6.1
DATA_PATH = r"C:\Users\guill\Desktop\Data Challenge QRT\Data-Challenge-Prediction-de-Survie"

clinical_train = pd.read_csv(f"{DATA_PATH}\\X_train\\clinical_train.csv")
target_train = pd.read_csv(f"{DATA_PATH}\\target_train.csv")
clinical_test = pd.read_csv(f"{DATA_PATH}\\X_test\\clinical_test.csv")
molecular_train = pd.read_csv(f"{DATA_PATH}\\X_train\\molecular_train.csv")
molecular_test = pd.read_csv(f"{DATA_PATH}\\X_test\\molecular_test.csv")

print("✓ Data loaded")

✓ Data loaded


In [4]:
# Feature engineering functions (condensed from V6.1)
def create_cytogenetic_features(clinical_df):
    cyto_features = pd.DataFrame(index=clinical_df['ID'])
    cyto_col = clinical_df.set_index('ID')['CYTOGENETICS'].fillna('')
    cyto_features['cyto_del_count'] = cyto_col.str.count(r'del\(')
    cyto_features['cyto_has_del'] = (cyto_features['cyto_del_count'] > 0).astype(int)
    cyto_features['cyto_transloc_count'] = cyto_col.str.count(r't\(')
    cyto_features['cyto_has_transloc'] = (cyto_features['cyto_transloc_count'] > 0).astype(int)
    cyto_features['cyto_inv_count'] = cyto_col.str.count(r'inv\(')
    cyto_features['cyto_has_inv'] = (cyto_features['cyto_inv_count'] > 0).astype(int)
    cyto_features['cyto_gain_count'] = cyto_col.str.count(r'\+')
    cyto_features['cyto_has_gain'] = (cyto_features['cyto_gain_count'] > 0).astype(int)
    cyto_features['cyto_loss_count'] = cyto_col.str.count(r'-[0-9XY]')
    cyto_features['cyto_has_loss'] = (cyto_features['cyto_loss_count'] > 0).astype(int)
    cyto_features['cyto_other_count'] = cyto_col.str.count(r'add\(|ins\(|dup\(')
    cyto_features['cyto_total_anomalies'] = (
        cyto_features['cyto_del_count'] + cyto_features['cyto_transloc_count'] + 
        cyto_features['cyto_inv_count'] + cyto_features['cyto_gain_count'] + 
        cyto_features['cyto_loss_count'] + cyto_features['cyto_other_count']
    )
    cyto_features['cyto_normal'] = cyto_col.str.match(r'^46,(xx|xy)(\[\d+\])?$', case=False).astype(int)
    cyto_features['cyto_complex'] = (
        (cyto_features['cyto_total_anomalies'] >= 3) | 
        cyto_col.str.contains('complex', case=False, na=False)
    ).astype(int)
    chromosomes = [str(i) for i in range(1, 23)] + ['X', 'Y']
    for chrom in chromosomes:
        pattern = rf'(\b|[,\(]){chrom}([,;:\)\[]|[pq])'
        cyto_features[f'cyto_chr{chrom}_affected'] = cyto_col.str.contains(
            pattern, case=False, na=False, regex=True
        ).astype(int)
    cyto_features['cyto_monosomy7'] = cyto_col.str.contains(r'-7[^0-9]|^45.*-7', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_trisomy8'] = cyto_col.str.contains(r'\+8[^0-9]|^47.*\+8', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_del5q'] = cyto_col.str.contains(r'del\(5\)\(q', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_del20q'] = cyto_col.str.contains(r'del\(20\)\(q', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_chr3_abnormal'] = cyto_col.str.contains(r'(del|t|inv)\(3[;,:\)]', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_chr7_abnormal'] = cyto_col.str.contains(r'(del|t|inv)\(7[;,:\)]', case=False, na=False, regex=True).astype(int)
    return cyto_features.fillna(0)

def create_molecular_features(molecular_df, patient_ids, top_n_genes=20):
    mol_features = pd.DataFrame({'ID': patient_ids})
    mutation_counts = molecular_df.groupby('ID').size().to_frame('mutation_count_total')
    mol_features = mol_features.merge(mutation_counts, on='ID', how='left')
    vaf_stats = molecular_df.groupby('ID')['VAF'].agg([
        ('vaf_mean', 'mean'), ('vaf_max', 'max'), ('vaf_sum', 'sum')
    ]).reset_index()
    mol_features = mol_features.merge(vaf_stats, on='ID', how='left')
    effect_counts = molecular_df.groupby(['ID', 'EFFECT']).size().unstack(fill_value=0)
    effect_counts.columns = [f'effect_{col}' for col in effect_counts.columns]
    mol_features = mol_features.merge(effect_counts.reset_index(), on='ID', how='left')
    top_genes_list = molecular_df['GENE'].value_counts().head(top_n_genes).index.tolist()
    for gene in top_genes_list:
        gene_mutations = molecular_df[molecular_df['GENE'] == gene].groupby('ID').size()
        mol_features[f'gene_{gene}_count'] = mol_features['ID'].map(gene_mutations)
        gene_present = molecular_df[molecular_df['GENE'] == gene]['ID'].unique()
        mol_features[f'gene_{gene}_present'] = mol_features['ID'].isin(gene_present).astype(int)
    feature_cols = [col for col in mol_features.columns if col != 'ID']
    mol_features[feature_cols] = mol_features[feature_cols].fillna(0)
    return mol_features.set_index('ID')

def create_advanced_features(X_df):
    X_adv = X_df.copy()
    X_adv['blast_to_wbc'] = X_adv['BM_BLAST'] / (X_adv['WBC'] + 1)
    X_adv['monocyte_ratio'] = X_adv['MONOCYTES'] / (X_adv['WBC'] + 1)
    X_adv['anc_ratio'] = X_adv['ANC'] / (X_adv['WBC'] + 1)
    X_adv['platelet_to_blast'] = X_adv['PLT'] / (X_adv['BM_BLAST'] + 1)
    X_adv['hb_to_plt'] = X_adv['HB'] / (X_adv['PLT'] + 1)
    X_adv['vaf_mutation_burden'] = X_adv['vaf_mean'] * X_adv['mutation_count_total']
    X_adv['vaf_per_mutation'] = X_adv['vaf_sum'] / (X_adv['mutation_count_total'] + 1)
    X_adv['cytogenetic_risk_score'] = (
        X_adv['cyto_complex'] * 3 + X_adv['cyto_chr7_affected'] * 2 + 
        X_adv['cyto_del5q'] * 1.5 + X_adv['cyto_loss_count'] * 0.5
    )
    X_adv['blast_cytogenetic_risk'] = X_adv['BM_BLAST'] * X_adv['cytogenetic_risk_score']
    X_adv['blast_to_mutation'] = X_adv['BM_BLAST'] / (X_adv['mutation_count_total'] + 1)
    X_adv['wbc_plt_index'] = X_adv['WBC'] * X_adv['PLT'] / 1000
    X_adv['blast_hb_ratio'] = X_adv['BM_BLAST'] / (X_adv['HB'] + 1)
    X_adv['monocyte_blast_ratio'] = X_adv['MONOCYTES'] / (X_adv['BM_BLAST'] + 1)
    X_adv['mutation_per_vaf'] = X_adv['mutation_count_total'] / (X_adv['vaf_mean'] + 0.01)
    X_adv['cyto_anomaly_density'] = X_adv['cyto_total_anomalies'] / (X_adv['cyto_total_anomalies'].max() + 1)
    X_adv['blast_mutation_interaction'] = X_adv['BM_BLAST'] * X_adv['mutation_count_total']
    X_adv['blast_vaf_interaction'] = X_adv['BM_BLAST'] * X_adv['vaf_mean']
    X_adv['blast_cyto_complex'] = X_adv['BM_BLAST'] * X_adv['cyto_complex']
    X_adv['tp53_blast'] = X_adv['gene_TP53_present'] * X_adv['BM_BLAST']
    X_adv['runx1_mutation_burden'] = X_adv['gene_RUNX1_count'] * X_adv['mutation_count_total']
    X_adv['nras_vaf'] = X_adv['gene_NRAS_present'] * X_adv['vaf_mean']
    X_adv['cyto_mutation_interaction'] = X_adv['cyto_total_anomalies'] * X_adv['mutation_count_total']
    X_adv['chr7_blast'] = X_adv['cyto_chr7_affected'] * X_adv['BM_BLAST']
    X_adv['vaf_cyto_burden'] = X_adv['vaf_sum'] * X_adv['cyto_total_anomalies']
    X_adv['vaf_tp53'] = X_adv['vaf_mean'] * X_adv['gene_TP53_present']
    X_adv['log_wbc'] = np.log1p(X_adv['WBC'])
    X_adv['log_plt'] = np.log1p(X_adv['PLT'])
    X_adv['log_blast'] = np.log1p(X_adv['BM_BLAST'])
    X_adv['log_mutation_count'] = np.log1p(X_adv['mutation_count_total'])
    X_adv['log_vaf_sum'] = np.log1p(X_adv['vaf_sum'])
    return X_adv

# Create base features
cyto_features_train = create_cytogenetic_features(clinical_train)
train_patient_ids = clinical_train['ID'].unique()
mol_features_train = create_molecular_features(molecular_train, train_patient_ids)

target_clean = target_train.dropna(subset=['OS_YEARS', 'OS_STATUS']).copy()
target_clean['OS_STATUS'] = target_clean['OS_STATUS'].astype(bool)
target_clean = target_clean.set_index('ID')

clinical_train_clean = clinical_train[clinical_train['ID'].isin(target_clean.index)].copy()
clinical_train_clean = clinical_train_clean.set_index('ID').loc[target_clean.index]

numeric_features = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'HB', 'PLT']
X_numeric = clinical_train_clean[numeric_features].copy()
center_encoded = pd.get_dummies(clinical_train_clean['CENTER'], prefix='CENTER', drop_first=True)
X_clinical = pd.concat([X_numeric, center_encoded], axis=1)

mol_features_train_aligned = mol_features_train.reindex(X_clinical.index, fill_value=0)
cyto_features_train_aligned = cyto_features_train.reindex(X_clinical.index, fill_value=0)
X_base = pd.concat([X_clinical, mol_features_train_aligned, cyto_features_train_aligned], axis=1)
X_base = create_advanced_features(X_base)

print(f"✓ Base features created: {X_base.shape[1]} features")

✓ Base features created: 162 features


## 4. Get Top Features (Quick Baseline)

In [5]:
print("="*60)
print("QUICK BASELINE TO GET TOP FEATURES")
print("="*60)

# Impute
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    imputer.fit_transform(X_base),
    index=X_base.index,
    columns=X_base.columns
)

y_surv = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_clean)

# Split
X_train_base, X_val_base, y_train, y_val = train_test_split(
    X_imputed, y_surv, test_size=0.3, random_state=42,
    stratify=target_clean['OS_STATUS'].astype(int)
)

# Train quick model
y_train_xgb = y_train['OS_YEARS'].copy()
y_train_xgb[~y_train['OS_STATUS']] = -y_train_xgb[~y_train['OS_STATUS']]

dtrain_quick = xgb.DMatrix(X_train_base, label=y_train_xgb)

model_quick = xgb.train(
    BEST_XGB_PARAMS,
    dtrain_quick,
    num_boost_round=100,
    verbose_eval=False
)

# Get top features
importance_dict = model_quick.get_score(importance_type='gain')
importance_df = pd.DataFrame([
    {'feature': k, 'importance': v} 
    for k, v in importance_dict.items()
]).sort_values('importance', ascending=False)

top_5_features = importance_df.head(5)['feature'].tolist()
top_10_features = importance_df.head(10)['feature'].tolist()

print(f"\n✓ Top 5 features: {top_5_features}")
print(f"✓ Top 10 features: {top_10_features[:5]}...")

QUICK BASELINE TO GET TOP FEATURES

✓ Top 5 features: ['cyto_mutation_interaction', 'vaf_cyto_burden', 'blast_mutation_interaction', 'vaf_tp53', 'blast_cytogenetic_risk']
✓ Top 10 features: ['cyto_mutation_interaction', 'vaf_cyto_burden', 'blast_mutation_interaction', 'vaf_tp53', 'blast_cytogenetic_risk']...


## 5. Quick Win #1: Polynomial Features

In [7]:
print("="*60)
print("QUICK WIN #1: POLYNOMIAL FEATURES (TOP 5)")
print("="*60)

# IMPORTANT: Use imputed data (from train/val split)
X_train_top5 = X_train_base[top_5_features].copy()

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly_array = poly.fit_transform(X_train_top5)

# Get feature names
poly_feature_names = poly.get_feature_names_out(top_5_features)
X_poly_train = pd.DataFrame(X_poly_array, index=X_train_top5.index, columns=poly_feature_names)

# Remove original features (already in X_train_base)
poly_new_features = [f for f in poly_feature_names if f not in top_5_features]
X_poly_train_new = X_poly_train[poly_new_features]

# Apply to validation set
X_val_top5 = X_val_base[top_5_features].copy()
X_poly_val_array = poly.transform(X_val_top5)
X_poly_val = pd.DataFrame(X_poly_val_array, index=X_val_top5.index, columns=poly_feature_names)
X_poly_val_new = X_poly_val[poly_new_features]

print(f"\n✓ Created {len(poly_new_features)} polynomial features")
print(f"  Examples: {poly_new_features[:5]}")

QUICK WIN #1: POLYNOMIAL FEATURES (TOP 5)

✓ Created 15 polynomial features
  Examples: ['cyto_mutation_interaction^2', 'cyto_mutation_interaction vaf_cyto_burden', 'cyto_mutation_interaction blast_mutation_interaction', 'cyto_mutation_interaction vaf_tp53', 'cyto_mutation_interaction blast_cytogenetic_risk']


## 6. Quick Win #2: Feature Interactions

In [8]:
print("="*60)
print("QUICK WIN #2: FEATURE INTERACTIONS (TOP 10 × TOP 10)")
print("="*60)

# Use imputed train data
X_interactions_train = pd.DataFrame(index=X_train_base.index)
X_interactions_val = pd.DataFrame(index=X_val_base.index)

# All pairwise interactions
for feat1, feat2 in combinations(top_10_features, 2):
    # Train
    X_interactions_train[f'{feat1}_×_{feat2}'] = X_train_base[feat1] * X_train_base[feat2]
    X_interactions_train[f'{feat1}_÷_{feat2}'] = X_train_base[feat1] / (X_train_base[feat2] + 1)
    
    # Val
    X_interactions_val[f'{feat1}_×_{feat2}'] = X_val_base[feat1] * X_val_base[feat2]
    X_interactions_val[f'{feat1}_÷_{feat2}'] = X_val_base[feat1] / (X_val_base[feat2] + 1)

print(f"\n✓ Created {X_interactions_train.shape[1]} interaction features")
print(f"  From {len(top_10_features)} top features")
print(f"  {int(len(top_10_features)*(len(top_10_features)-1)/2)} pairs × 2 operations")

QUICK WIN #2: FEATURE INTERACTIONS (TOP 10 × TOP 10)

✓ Created 90 interaction features
  From 10 top features
  45 pairs × 2 operations


## 7. Combine All Features

In [9]:
print("="*60)
print("COMBINING ALL FEATURES")
print("="*60)

# Combine: Base + Polynomial + Interactions
X_train_enh = pd.concat([X_train_base, X_poly_train_new, X_interactions_train], axis=1)
X_val_enh = pd.concat([X_val_base, X_poly_val_new, X_interactions_val], axis=1)

print(f"\n✓ Feature counts:")
print(f"  Base features:        {X_train_base.shape[1]}")
print(f"  + Polynomial:         {X_poly_train_new.shape[1]}")
print(f"  + Interactions:       {X_interactions_train.shape[1]}")
print(f"  = Total:              {X_train_enh.shape[1]}")

print(f"\n✓ Train: {X_train_enh.shape}, Val: {X_val_enh.shape}")

COMBINING ALL FEATURES

✓ Feature counts:
  Base features:        162
  + Polynomial:         15
  + Interactions:       90
  = Total:              267

✓ Train: (2221, 267), Val: (952, 267)


## 8. Quick Win #3: Bagging Ensemble

In [10]:
print("="*60)
print(f"QUICK WIN #3: BAGGING ENSEMBLE ({N_BAGGING_MODELS} MODELS)")
print("="*60)

# Prepare data
y_val_xgb = y_val['OS_YEARS'].copy()
y_val_xgb[~y_val['OS_STATUS']] = -y_val_xgb[~y_val['OS_STATUS']]

dtrain_enh = xgb.DMatrix(X_train_enh, label=y_train_xgb)
dval_enh = xgb.DMatrix(X_val_enh, label=y_val_xgb)

# Train multiple models with different seeds
models = []
predictions_val = []

for seed in range(N_BAGGING_MODELS):
    print(f"\nTraining model {seed+1}/{N_BAGGING_MODELS}...")
    
    params_seed = BEST_XGB_PARAMS.copy()
    params_seed['seed'] = seed
    
    model = xgb.train(
        params_seed,
        dtrain_enh,
        num_boost_round=500,
        evals=[(dval_enh, 'val')],
        early_stopping_rounds=50,
        verbose_eval=False
    )
    
    models.append(model)
    
    # Predict
    y_pred = model.predict(dval_enh)
    predictions_val.append(y_pred)
    
    # Individual C-index
    c_index_single = concordance_index_censored(
        y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred
    )[0]
    
    print(f"  Model {seed+1} C-index: {c_index_single:.4f}")

# Ensemble prediction (average)
y_pred_ensemble = np.mean(predictions_val, axis=0)

c_index_ensemble = concordance_index_censored(
    y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred_ensemble
)[0]

print(f"\n✓ Ensemble C-index: {c_index_ensemble:.4f}")

QUICK WIN #3: BAGGING ENSEMBLE (5 MODELS)

Training model 1/5...
  Model 1 C-index: 0.7299

Training model 2/5...
  Model 2 C-index: 0.7318

Training model 3/5...
  Model 3 C-index: 0.7300

Training model 4/5...
  Model 4 C-index: 0.7298

Training model 5/5...
  Model 5 C-index: 0.7291

✓ Ensemble C-index: 0.7325


## 9. Performance Comparison

In [12]:
# Baseline with base features only
y_pred_baseline_val = model_quick.predict(xgb.DMatrix(X_val_base))
c_index_baseline = concordance_index_censored(
    y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred_baseline_val
)[0]

# Single model C-index
c_index_single = concordance_index_censored(
    y_val['OS_STATUS'], y_val['OS_YEARS'], predictions_val[0]
)[0]

print("="*60)
print("PERFORMANCE COMPARISON")
print("="*60)

comparison = pd.DataFrame([
    {'Model': 'V4 RSF Baseline', 'Features': 90, 'C-index': 0.7404},
    {'Model': 'XGB Base Features', 'Features': X_train_base.shape[1], 'C-index': c_index_baseline},
    {'Model': 'Single Model (enhanced)', 'Features': X_train_enh.shape[1], 'C-index': c_index_single},
    {'Model': 'V6.2 Ensemble (Final)', 'Features': X_train_enh.shape[1], 'C-index': c_index_ensemble}
])

print("\n" + comparison.to_string(index=False))

improvement = c_index_ensemble - 0.7404
improvement_vs_baseline = c_index_ensemble - c_index_baseline

print(f"\n📊 Improvements:")
print(f"  vs V4 RSF:           {improvement:+.4f}")
print(f"  vs XGB Baseline:     {improvement_vs_baseline:+.4f}")
print(f"\n🎯 Quick Wins Impact:")
print(f"  Polynomial features: ~{(X_poly_train_new.shape[1]/X_train_enh.shape[1])*100:.1f}% of features")
print(f"  Interactions:        ~{(X_interactions_train.shape[1]/X_train_enh.shape[1])*100:.1f}% of features")
print(f"  Bagging:             {N_BAGGING_MODELS} models averaged")

PERFORMANCE COMPARISON

                  Model  Features  C-index
        V4 RSF Baseline        90 0.740400
      XGB Base Features       162 0.733386
Single Model (enhanced)       267 0.729853
  V6.2 Ensemble (Final)       267 0.732493

📊 Improvements:
  vs V4 RSF:           -0.0079
  vs XGB Baseline:     -0.0009

🎯 Quick Wins Impact:
  Polynomial features: ~5.6% of features
  Interactions:        ~33.7% of features
  Bagging:             5 models averaged


In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(12, 6))

models_viz = ['V4\nRSF', 'XGB\nBase', 'XGB\nEnhanced\n(Single)', 'V6.2\nEnsemble']
scores = [0.7404, c_index_baseline, 
          concordance_index_censored(y_val['OS_STATUS'], y_val['OS_YEARS'], predictions_val[0])[0],
          c_index_ensemble]
colors = ['#808080', '#2E86AB', '#F18F01', '#27AE60']

bars = ax.bar(models_viz, scores, color=colors, alpha=0.7, edgecolor='black', linewidth=2)

for bar, score in zip(bars, scores):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{score:.4f}',
            ha='center', va='bottom', fontsize=13, fontweight='bold')

ax.axhline(y=0.74, color='orange', linestyle='--', linewidth=2, alpha=0.5, label='Baseline')
ax.axhline(y=0.75, color='green', linestyle='--', linewidth=2, alpha=0.5, label='Target: 0.75')
ax.axhline(y=0.76, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Goal: 0.76')

ax.set_ylabel('C-index (Validation)', fontsize=12)
ax.set_title('V6.2 Quick Wins Performance', fontsize=14, fontweight='bold')
ax.set_ylim([0.72, max(0.77, c_index_ensemble + 0.01)])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 10. Test Set & Submission

In [ ]:
print("="*60)
print("PREPARING TEST SET")
print("="*60)

# Create base features for test (same pipeline)
cyto_features_test = create_cytogenetic_features(clinical_test)
test_patient_ids = clinical_test['ID'].unique()
mol_features_test = create_molecular_features(molecular_test, test_patient_ids)

clinical_test_indexed = clinical_test.set_index('ID')
X_numeric_test = clinical_test_indexed[numeric_features].copy()
center_encoded_test = pd.get_dummies(clinical_test_indexed['CENTER'], prefix='CENTER', drop_first=True)
for col in center_encoded.columns:
    if col not in center_encoded_test.columns:
        center_encoded_test[col] = 0
center_encoded_test = center_encoded_test[center_encoded.columns]
X_clinical_test = pd.concat([X_numeric_test, center_encoded_test], axis=1)

mol_features_test_aligned = mol_features_test.reindex(X_clinical_test.index, fill_value=0)
cyto_features_test_aligned = cyto_features_test.reindex(X_clinical_test.index, fill_value=0)
X_test_base = pd.concat([X_clinical_test, mol_features_test_aligned, cyto_features_test_aligned], axis=1)
X_test_base = create_advanced_features(X_test_base)

# Create polynomial features for test
X_test_top5 = X_test_base[top_5_features].copy()
X_test_poly_array = poly.transform(X_test_top5)
X_test_poly = pd.DataFrame(X_test_poly_array, index=X_test_top5.index, columns=poly_feature_names)
X_test_poly_new = X_test_poly[poly_new_features]

# Create interactions for test
X_test_interactions = pd.DataFrame(index=X_test_base.index)
for feat1, feat2 in combinations(top_10_features, 2):
    X_test_interactions[f'{feat1}_×_{feat2}'] = X_test_base[feat1] * X_test_base[feat2]
    X_test_interactions[f'{feat1}_÷_{feat2}'] = X_test_base[feat1] / (X_test_base[feat2] + 1)

# Combine
X_test_enhanced = pd.concat([X_test_base, X_test_poly_new, X_test_interactions], axis=1)

# Ensure same columns
for col in X_enhanced_imputed.columns:
    if col not in X_test_enhanced.columns:
        X_test_enhanced[col] = 0
X_test_enhanced = X_test_enhanced[X_enhanced_imputed.columns]

X_test_imputed = pd.DataFrame(
    imputer_enhanced.transform(X_test_enhanced),
    index=X_test_enhanced.index,
    columns=X_test_enhanced.columns
)

print(f"✓ Test set prepared: {X_test_imputed.shape}")

In [ ]:
print("\nRetraining ensemble on full dataset...")

# Full training set
y_full_xgb = target_clean['OS_YEARS'].copy()
y_full_xgb[~target_clean['OS_STATUS']] = -y_full_xgb[~target_clean['OS_STATUS']]
dfull = xgb.DMatrix(X_enhanced_imputed, label=y_full_xgb)

# Retrain all models
models_final = []
for seed in range(N_BAGGING_MODELS):
    params_seed = BEST_XGB_PARAMS.copy()
    params_seed['seed'] = seed
    
    model = xgb.train(
        params_seed,
        dfull,
        num_boost_round=models[seed].best_iteration,
        verbose_eval=False
    )
    models_final.append(model)

print(f"✓ {N_BAGGING_MODELS} models retrained")

In [ ]:
# Generate ensemble predictions
dtest = xgb.DMatrix(X_test_imputed)

predictions_test = [model.predict(dtest) for model in models_final]
y_pred_test_ensemble = np.mean(predictions_test, axis=0)

# Convert to risk scores
min_pred = y_pred_test_ensemble.min()
max_pred = y_pred_test_ensemble.max()
risk_scores = 1 - (y_pred_test_ensemble - min_pred) / (max_pred - min_pred)

submission = pd.DataFrame({
    'ID': X_test_imputed.index,
    'risk_score': risk_scores
})

submission_path = f"{DATA_PATH}\\submission_v6.2_quickwins.csv"
submission.to_csv(submission_path, index=False)

print("="*60)
print("SUBMISSION GENERATED")
print("="*60)
print(f"File: {submission_path}")
print(f"Predictions: {len(submission)}")
print(f"\n📊 Risk Scores (0-1):")
print(submission['risk_score'].describe())

## 11. Summary

In [ ]:
print("="*60)
print("VERSION 6.2 - QUICK WINS SUMMARY")
print("="*60)

print("\n🎯 QUICK WINS APPLIED:")
print(f"  1. Polynomial Features (top 5):  {X_poly_new.shape[1]} features")
print(f"  2. Interactions (top 10 × 10):   {X_interactions.shape[1]} features")
print(f"  3. Bagging Ensemble:             {N_BAGGING_MODELS} models")

print(f"\n📊 FEATURES:")
print(f"  Base:     {X_base.shape[1]}")
print(f"  Enhanced: {X_enhanced.shape[1]} (+{X_enhanced.shape[1] - X_base.shape[1]})")

print("\n📈 PERFORMANCE:")
print(f"  XGB Baseline:  {c_index_baseline:.4f}")
print(f"  V6.2 Ensemble: {c_index_ensemble:.4f}")

print("\n🚀 IMPROVEMENT:")
print(f"  vs V4 RSF:        {improvement:+.4f}")
print(f"  vs XGB Baseline:  {improvement_vs_baseline:+.4f}")

if c_index_ensemble >= 0.76:
    print("\n🏆 GOAL EXCEEDED! (C-index ≥ 0.76)")
elif c_index_ensemble >= 0.75:
    print("\n🎯 TARGET REACHED! (C-index ≥ 0.75)")
elif c_index_ensemble > c_index_baseline:
    print("\n✅ IMPROVED!")

print("\n✅ OUTPUT:")
print(f"  Submission: submission_v6.2_quickwins.csv")
print(f"  Ready for submission!")